In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [ ]:
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

dados = pd.read_csv(path + "/creditcard.csv")

In [ ]:
dados.head()

In [ ]:
dados.shape

In [ ]:
dados.info()

In [ ]:
dados["Class"].value_counts()

O dataset possui 284.807 transações financeiras e 31 atributos. A variável Class representa a classificação da transação, sendo 0 para transações legítimas e 1 para fraudes. Observa-se um forte desbalanceamento entre as classes, com uma quantidade muito reduzida de registros fraudulentos.

In [ ]:
print("Valores nulos por coluna:")
print(dados.isnull().sum())

In [ ]:
print("\nEstatísticas gerais:")
print(dados.describe())

In [ ]:
print("\nQuantidade de transações por classe:")
print(dados["Class"].value_counts())

print("\nPorcentagem de cada classe:")
print((dados["Class"].value_counts(normalize=True) * 100).round(3))

In [ ]:
print("\nResumo da coluna Amount:")
print(dados["Amount"].describe())

In [ ]:
dadosFraudes = dados[dados["Class"] == 1]
dadosNormais = dados[dados["Class"] == 0]

print("\nResumo das transações fraudulentas:")
print(dadosFraudes["Amount"].describe())

print("\nResumo das transações normais:")
print(dadosNormais["Amount"].describe())


In [ ]:
plt.figure(figsize=(10,6))

plt.hist(
    dados["Amount"],
    bins=50,
    edgecolor="black",
    alpha=0.7
)

plt.axvline(
    dados["Amount"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Média = {dados['Amount'].mean():.2f}"
)

plt.title("Distribuição dos valores das transações")
plt.xlabel("Valor da transação")
plt.ylabel("Quantidade")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.boxplot(
    dados["Amount"],
    vert=False
)

plt.title("Boxplot dos valores das transações")
plt.xlabel("Valor")

plt.show()


In [ ]:
plt.figure(figsize=(6,4))

dados["Class"].value_counts().plot(
    kind="bar"
)

plt.title("Quantidade de transações por classe")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks([0,1],["Normal","Fraude"], rotation=0)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.boxplot(
    [
        dadosNormais["Amount"],
        dadosFraudes["Amount"]
    ],
    labels=["Normal","Fraude"]
)

plt.title("Comparação dos valores das transações")
plt.ylabel("Valor")

plt.show()

Conclusão da análise exploratória

O conjunto de dados possui 284.807 transações financeiras e 31 atributos.
Não foram encontrados valores nulos.
O dataset é altamente desbalanceado, contendo aproximadamente 99,83% de transações legítimas e apenas 0,17% de fraudes.
A variável Amount apresenta distribuição assimétrica, com predominância de transações de baixo valor e presença de valores extremos (outliers).
As estatísticas indicam diferenças entre os valores das transações normais e fraudulentas, justificando a aplicação de um algoritmo de detecção de anomalias.


In [ ]:
X = dados.drop(columns=["Class"])

y = dados["Class"]

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
modelo = IsolationForest(
    n_estimators=100,
    contamination=0.002,
    random_state=42
)

modelo.fit(X_scaled)

In [ ]:
predicoes = modelo.predict(X_scaled)

In [ ]:
predicoes = np.where(predicoes == -1, 1, 0)

In [ ]:
print("Anomalias detectadas:")

print(pd.Series(predicoes).value_counts())

In [ ]:
matriz = confusion_matrix(y, predicoes)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Normal", "Fraude"]
)

disp.plot(cmap="Blues")

plt.title("Matriz de Confusão")

plt.show()

In [ ]:
print(classification_report(y, predicoes))

In [ ]:
anomalias = dados[predicoes == 1]

normais = dados[predicoes == 0]

plt.figure(figsize=(10,5))

plt.boxplot(
    [
        normais["Amount"],
        anomalias["Amount"]
    ],
    tick_labels=["Normal","Anomalia"]
)

plt.title("Comparação dos Valores Detectados")

plt.ylabel("Valor")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

pd.Series(predicoes).value_counts().plot(
    kind="bar"
)

plt.xticks(
    [0,1],
    ["Normal","Anomalia"],
    rotation=0
)

plt.title("Quantidade de Transações Detectadas")

plt.ylabel("Quantidade")

plt.show()

# Conclusão

Neste projeto foi desenvolvido um modelo de detecção de anomalias utilizando o algoritmo Isolation Forest para identificar possíveis fraudes em transações financeiras.

Após a análise exploratória dos dados, verificou-se que o conjunto de dados é altamente desbalanceado, contendo aproximadamente 99,83% de transações legítimas.

O modelo foi treinado de forma não supervisionada, sem utilizar a variável alvo durante o treinamento. Posteriormente, as anomalias detectadas foram comparadas com os rótulos reais do dataset por meio da matriz de confusão e das métricas de avaliação.

Embora o Isolation Forest consiga identificar parte das transações fraudulentas, ainda ocorrem falsos positivos e fraudes não detectadas, demonstrando que a detecção de anomalias é um problema complexo e pode ser aprimorada com ajuste de parâmetros, engenharia de atributos ou utilização de outros algoritmos especializados.